# COSMOS

The goal is to find the tract and patch of a DDF field.
As such , it is possible to make a Data Query SQL request based on tract and patch.
We use DP2 to get the Rubin Skymapping.

- author : Sylvie Dagoret-Campagne
- creation-date : 2026-10-25

In [ ]:
import warnings

warnings.filterwarnings(
    "error",
    category=FutureWarning,
    message=".*Butler.collections.*",
)

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

import numpy as np
import lsst.geom as geom

## Configuration of Butler

In [ ]:
from lsst.daf.butler import Butler

butler_url = "dp2"
repo = "dp2"

collection = "dp2"
instrument = "LSSTCam"
skymapName = "lsst_cells_v2"
butler = Butler(repo)

In [ ]:
# Derived configuration (do not edit)
where_clause = f"instrument = '{instrument}'"


# Initialise the Butler repository
butler = Butler(repo, collections=collection)
registry = butler.registry
print(f"Butler initialised | repo: {repo}")
print(f"Collection : {collection[-1]}")

In [ ]:
skymap = butler.get("skyMap", skymap=skymapName, collections=collection)

## Define DDF coordinates

In [ ]:
lsstcam_fields = {}

# =========================
# Deep Drilling Fields (LSSTCam)
# =========================
lsstcam_fields["COSMOS"] = {"field_name": "COSMOS Deep Drilling Field", "ra": 150.11, "dec": 2.23}
lsstcam_fields["XMM-LSS"] = {"field_name": "XMM-LSS Deep Drilling Field", "ra": 35.57, "dec": -4.82}
lsstcam_fields["ELAIS-S1"] = {"field_name": "ELAIS-S1 Deep Drilling Field", "ra": 9.45, "dec": -44.02}
lsstcam_fields["ECDFS"] = {"field_name": "Extended Chandra Deep Field South", "ra": 52.98, "dec": -28.12}
# Euclid Deep Field South = 2 pointings LSSTCam
lsstcam_fields["EDFS_a"] = {"field_name": "Euclid Deep Field South (pointing a)", "ra": 58.90, "dec": -49.32}
lsstcam_fields["EDFS_b"] = {"field_name": "Euclid Deep Field South (pointing b)", "ra": 63.60, "dec": -47.60}


In [ ]:
lsstcam_fields["COSMOS"]["ra"]

## Find collections

In [ ]:
collections = list(butler.collections.query(f"*"))

In [ ]:
for collection in collections:
    if "DP2" in collection:
        print(collection)

print(f"\nNombre de collections : {len(collections)}")


## Find Tract and Patch

In [ ]:
import numpy as np
import lsst.geom as geom

ra_center = lsstcam_fields["COSMOS"]["ra"]
dec_center = lsstcam_fields["COSMOS"]["dec"]
field_size = 3.0  # degrés

ras = np.linspace(
    ra_center - field_size / 2,
    ra_center + field_size / 2,
    21,
)

decs = np.linspace(
    dec_center - field_size / 2,
    dec_center + field_size / 2,
    21,
)

coord_list = [
    geom.SpherePoint(ra * geom.degrees, dec * geom.degrees)
    for ra in ras
    for dec in decs
]

tract_patch_list = skymap.findTractPatchList(coord_list)

for tract_info, patch_infos in tract_patch_list:
    patch_ids = sorted({
        patch.getSequentialIndex()
        for patch in patch_infos
    })

    print(
        f"Tract {tract_info.tract_id}: "
        f"patches {patch_ids}"
    )

In [ ]:
def patch_boundary_sky(tract_info, bbox, n=30):
    """Contour d'une bbox de patch, transformé en RA/Dec."""
    wcs = tract_info.getWcs()

    xmin, ymin = bbox.getMinX(), bbox.getMinY()
    xmax, ymax = bbox.getMaxX(), bbox.getMaxY()

    # Échantillonnage des quatre côtés du rectangle
    x = np.linspace(xmin, xmax, n)
    y = np.linspace(ymin, ymax, n)

    pixels = (
        [(xx, ymin) for xx in x]
        + [(xmax, yy) for yy in y[1:]]
        + [(xx, ymax) for xx in x[-2::-1]]
        + [(xmin, yy) for yy in y[-2:0:-1]]
    )

    sky = [
        wcs.pixelToSky(geom.Point2D(float(xx), float(yy)))
        for xx, yy in pixels
    ]

    return (
        np.array([p.getRa().asDegrees() for p in sky]),
        np.array([p.getDec().asDegrees() for p in sky]),
    )


In [ ]:
# Centre du DDF COSMOS
ra0, dec0 = ra_center , dec_center
half_size = field_size/2.   # demi-largeur en degrés

In [ ]:
# Grille de points pour identifier les tracts concernés
ras = np.linspace(ra0 - half_size, ra0 + half_size, 31)
decs = np.linspace(dec0 - half_size, dec0 + half_size, 31)

tract_ids = set()

for ra in ras:
    for dec in decs:
        coord = geom.SpherePoint(
            ra * geom.degrees,
            dec * geom.degrees,
        )
        tract_ids.add(skymap.findTract(coord).getId())

print("Tracts détectés :", sorted(tract_ids))

In [ ]:
# Figure
fig, ax = plt.subplots(figsize=(10, 7))

for tract_info in skymap:
    if tract_info.getId() not in tract_ids:
        continue

    # Contour du tract dans le plan pixel du tract
    bbox = tract_info.getBBox()
    wcs = tract_info.getWcs()

    xmin, ymin = bbox.getMinX(), bbox.getMinY()
    xmax, ymax = bbox.getMaxX(), bbox.getMaxY()

    n = 50
    x = np.linspace(xmin, xmax, n)
    y = np.linspace(ymin, ymax, n)

    pixels = (
        [(xx, ymin) for xx in x]
        + [(xmax, yy) for yy in y[1:]]
        + [(xx, ymax) for xx in x[-2::-1]]
        + [(xmin, yy) for yy in y[-2:0:-1]]
    )

    sky = [
        wcs.pixelToSky(geom.Point2D(float(xx), float(yy)))
        for xx, yy in pixels
    ]

    ra = np.array([p.getRa().asDegrees() for p in sky])
    dec = np.array([p.getDec().asDegrees() for p in sky])

    ax.plot(ra, dec, linewidth=2)

    # Identifiant du tract au centre
    center_sky = wcs.pixelToSky(
        geom.Point2D(
            (xmin + xmax) / 2,
            (ymin + ymax) / 2,
        )
    )

    ax.text(
        center_sky.getRa().asDegrees(),
        center_sky.getDec().asDegrees(),
        str(tract_info.getId()),
        fontsize=11,
        ha="center",
        va="center",
    )

# Limites du champ
ax.set_xlim(ra0 + half_size, ra0 - half_size)
ax.set_ylim(dec0 - half_size, dec0 + half_size)

ax.set_xlabel("Right Ascension (deg)", fontsize=12)
ax.set_ylabel("Declination (deg)", fontsize=12)
ax.set_title("Rubin SkyMap — COSMOS DDF : tracts", fontsize=14)

ax.tick_params(labelsize=10)
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

for tract_info in skymap:
    if tract_info.getId() not in tract_ids:
        continue

    for patch_info in tract_info:
        # InnerBBox : contour nominal sans recouvrement entre patches
        bbox = patch_info.getInnerBBox()

        ra, dec = patch_boundary_sky(tract_info, bbox)

        ax.plot(ra, dec, linewidth=0.8)

        # Identifiant du patch au centre de sa boîte
        cx = (bbox.getMinX() + bbox.getMaxX()) / 2
        cy = (bbox.getMinY() + bbox.getMaxY()) / 2
        sky_center = tract_info.getWcs().pixelToSky(
            geom.Point2D(cx, cy)
        )

        ax.text(
            sky_center.getRa().asDegrees(),
            sky_center.getDec().asDegrees(),
            f"{tract_info.getId()}_"
            f"{patch_info.getSequentialIndex()}",
            fontsize=6,
            ha="center",
            va="center",
        )

ax.set_xlim(ra0 + half_size, ra0 - half_size)
ax.set_ylim(dec0 - half_size, dec0 + half_size)
ax.set_xlabel("Right Ascension (deg)")
ax.set_ylabel("Declination (deg)")
ax.set_title("Rubin SkyMap — COSMOS DDF")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()